In [4]:
import pandas as pd

file_path = '../data/raw/listings_summary.csv'
df_raw = pd.read_csv(file_path)

colonnes_a_garder = [
    'id', 'neighbourhood_cleansed', 'latitude', 'longitude', 
    'property_type', 'room_type', 'accommodates', 'bathrooms_text', 
    'bedrooms', 'beds', 'minimum_nights', 'availability_365', 
    'number_of_reviews', 'number_of_reviews_ltm', 'review_scores_rating'
]
df_clean = df_raw[colonnes_a_garder].copy()

#Création de la cible "is_popular" (>= 12 avis sur l'année écoulée)
df_clean['is_popular'] = (df_clean['number_of_reviews_ltm'] >= 12).astype(int)
df_ml = df_clean.copy()

#Suppression des colonnes tricheuses (Data Leakage) et inutiles
colonnes_triche = ['id', 'number_of_reviews', 'number_of_reviews_ltm']
df_ml = df_ml.drop(columns=colonnes_triche)

#Nettoyage des salles de bain
df_ml['bathrooms'] = df_ml['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)
df_ml = df_ml.drop(columns=['bathrooms_text'])

#Remplissage des valeurs manquantes
df_ml['review_scores_rating'] = df_ml['review_scores_rating'].fillna(df_ml['review_scores_rating'].mean())
df_ml['bedrooms'] = df_ml['bedrooms'].fillna(1)
df_ml['beds'] = df_ml['beds'].fillna(1)
df_ml['bathrooms'] = df_ml['bathrooms'].fillna(1)

print(f"Dimensions finales : {df_ml.shape[0]} lignes, {df_ml.shape[1]} colonnes.")
print(f"Valeurs manquantes restantes : {df_ml.isnull().sum().sum()}")

#Aperçu du résultat final
df_ml.head()

KeyError: "['neighbourhood_cleansed', 'property_type', 'accommodates', 'bathrooms_text', 'bedrooms', 'beds', 'review_scores_rating'] not in index"

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

#On transforme les colonnes de texte en chiffres (0 ou 1)
df_encoded = pd.get_dummies(df_ml, columns=['neighbourhood_cleansed', 'property_type', 'room_type'])

# X = les critères de l'appartement
# y = ce qu'on veut prédire 
X = df_encoded.drop(columns=['is_popular'])
y = df_encoded['is_popular']

#On garde 20% des données de côté pour faire passer un examen final au modèle
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#On crée l'IA
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

#L'IA apprend en regardant les données d'entraînement
rf_model.fit(X_train, y_train)
print("Entraînement terminé !")

#L'IA passe son examen sur les 20% de données qu'elle n'a jamais vues
y_pred = rf_model.predict(X_test)

#On compare ses réponses avec la réalité
precision = accuracy_score(y_test, y_pred)
print(f"Précision des prédictions : {precision * 100:.2f} %")

print("\nTop 3 des secrets de la rentabilité :")
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(3))

--- 1. ENCODAGE DES TEXTES ---
--- 2. SÉPARATION DES DONNÉES (TRAIN / TEST) ---
--- 3. ENTRAÎNEMENT DU MODÈLE (RANDOM FOREST) ---
✅ Entraînement terminé !
--- 4. RÉSULTATS DE L'EXAMEN ---
🎯 Précision de tes prédictions : 84.39 %

🏆 Top 3 des secrets de la rentabilité (ce que l'IA regarde en premier) :
review_scores_rating    0.222393
availability_365        0.194547
latitude                0.150208
dtype: float64


In [ ]:
# On demande à l'IA de calculer la probabilité de succès (de 0 à 1) pour tous les logements
# On récupère la colonne [1] qui correspond à la probabilité d'être un "Top Performer"
df_clean['probabilite_succes'] = rf_model.predict_proba(X)[:, 1]

# On le met en format pourcentage pour que ce soit plus joli dans Power BI
df_clean['probabilite_succes'] = round(df_clean['probabilite_succes'] * 100, 1)

# On sauvegarde le tableau propre (avec le texte lisible) dans le dossier "processed"
chemin_sauvegarde = '../data/processed/airbnb_predictions_paris.csv'
df_clean.to_csv(chemin_sauvegarde, index=False)

print(f"Fichier parfait pour Power BI sauvegardé dans : {chemin_sauvegarde}")

print("\nTop 10 des futurs 'Bestsellers':")
# On affiche les appartements avec la plus haute probabilité
df_clean[['id', 'neighbourhood_cleansed', 'room_type', 'probabilite_succes']].sort_values(by='probabilite_succes', ascending=False).head(10)

--- 5. GÉNÉRATION DES PRÉDICTIONS ET EXPORT ---
✅ Fichier parfait pour Power BI sauvegardé dans : ../data/processed/airbnb_predictions_paris.csv

🏆 Top 10 des futurs 'Bestsellers' selon ton Intelligence Artificielle :


,id,neighbourhood_cleansed,room_type,probabilite_succes
69810,1228068090962171923,Élysée,Entire home/apt,100.0
23431,36678564,Entrepôt,Entire home/apt,100.0
53337,1049047487253252222,Vaugirard,Entire home/apt,100.0
19411,28355084,Buttes-Montmartre,Entire home/apt,100.0
52327,1033980056838374123,Bourse,Entire home/apt,100.0
70715,1252162775696675026,Élysée,Entire home/apt,100.0
40761,835426499318791179,Louvre,Entire home/apt,100.0
70373,1243975039875326978,Gobelins,Entire home/apt,100.0
65857,1170885284845639320,Élysée,Entire home/apt,100.0
66459,1178816543920259930,Luxembourg,Entire home/apt,100.0
